## Task 6: Asynchronous RAG Pipeline with Cross-Encoder Reranking
**Requires:** FastAPI, LangChain, FAISS, SentenceTransformers, and internet access to download the embedding/reranker models. None of that is available in this sandbox, so the cell below is the complete reference implementation, not executed.

In [5]:
!pip install fastapi sentence-transformers faiss-cpu -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 79.9 MB/s eta 0:00:00


In [6]:
!pip install faiss-cpu -q


In [7]:

import asyncio
from fastapi import FastAPI
from sentence_transformers import SentenceTransformer, CrossEncoder
import faiss

app = FastAPI()
bi_encoder = SentenceTransformer('all-MiniLM-L6-v2')
cross_encoder = CrossEncoder('cross-encoder/ms-marco-MiniLM-L-6-v2')

documents = [
    "Paris is the capital of France.",
    "The Eiffel Tower is located in Paris.",
    "Python is a popular programming language.",
    "Machine learning is a subset of artificial intelligence.",
    "The Great Wall of China is visible from low orbit.",
    "Tokyo is the capital of Japan.",
]
doc_embeddings = bi_encoder.encode(documents)
index = faiss.IndexFlatIP(doc_embeddings.shape[1])
index.add(doc_embeddings)

async def retrieve(query, top_k=4):
    q_emb = bi_encoder.encode([query])
    scores, idxs = index.search(q_emb, top_k)
    return [documents[i] for i in idxs[0]]

async def rerank(query, candidates, top_n=2):
    pairs = [[query, c] for c in candidates]
    scores = cross_encoder.predict(pairs)
    ranked = sorted(zip(candidates, scores), key=lambda x: -x[1])
    return [c for c, s in ranked[:top_n]]

@app.get("/query")
async def handle_query(q: str):
    candidates = await retrieve(q)
    top_docs = await rerank(q, candidates)
    return {"query": q, "context": top_docs}

result = await handle_query("What is the capital of France?")
print(result)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/1.33k [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/711k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

{'query': 'What is the capital of France?', 'context': ['Paris is the capital of France.', 'Tokyo is the capital of Japan.']}
